In [7]:
import pandas as pd

df = pd.read_csv("../data/raw/listings.csv.gz")
print(df.shape)

columnas = ["price", "room_type", "neighbourhood_cleansed", "accommodates",
            "bedrooms", "bathrooms", "number_of_reviews", 
            "review_scores_rating", "availability_365"]
df = df[columnas].copy()

#Limpiamos el dato de precio
df["price"] = df["price"].astype(str).str.replace(r"[$,]", "", regex=True).astype(float)

print(df.isna().sum())
df.describe()

(30468, 90)
price                      2108
room_type                     0
neighbourhood_cleansed        0
accommodates                  0
bedrooms                     24
bathrooms                 30468
number_of_reviews             0
review_scores_rating       4083
availability_365              0
dtype: int64


,price,accommodates,bedrooms,bathrooms,number_of_reviews,review_scores_rating,availability_365
count,2.836000e+04,30468.000000,30444.000000,0.0,30468.000000,26385.000000,30468.000000
mean,2.339918e+03,3.426546,1.528446,NaN,58.923920,4.756009,247.833891
std,1.409418e+04,2.366317,1.217048,NaN,95.335983,0.376259,119.096446
min,1.368000e+02,1.000000,0.000000,NaN,0.000000,1.000000,0.000000
25%,9.421975e+02,2.000000,1.000000,NaN,4.000000,4.700000,163.000000
50%,1.500000e+03,3.000000,1.000000,NaN,22.000000,4.840000,296.000000
75%,2.326625e+03,4.000000,2.000000,NaN,75.000000,4.950000,349.000000
max,1.826199e+06,16.000000,50.000000,NaN,1893.000000,5.000000,365.000000


In [12]:
import numpy as np 

#1.-Quitamos columnas vacías
df = df.drop(columns=["bathrooms"], errors="ignore")

#2.-Quitamos filas sin valor; precio, recamara, calificación
df = df.dropna(subset=["price", "bedrooms", "review_scores_rating"])

#3.-Quitamos prcios sin sentido
bajo, alto = df["price"].quantile([0.01, 0.99])
print(f"Rango de precios conservado: ${bajo:,.0f} a ${alto:,.0f}")
df = df[(df["price"] >= bajo) & (df["price"] <= alto)]

#4.-Generamos logaritmo para el modelo
df["log_price"] = np.log(df["price"])

print(df.shape)
df.describe()

Rango de precios conservado: $376 a $7,500
(23781, 9)


,price,accommodates,bedrooms,number_of_reviews,review_scores_rating,availability_365,log_price
count,23781.000000,23781.000000,23781.000000,23781.000000,23781.000000,23781.00000,23781.000000
mean,1773.676766,3.398343,1.475085,70.462638,4.758141,257.61423,7.297509
std,1166.492824,2.120582,0.939742,101.306016,0.365420,106.45117,0.601755
min,376.000000,1.000000,0.000000,1.000000,1.000000,0.00000,5.929589
25%,962.500000,2.000000,1.000000,9.000000,4.700000,179.00000,6.869534
50%,1486.000000,3.000000,1.000000,33.000000,4.840000,300.00000,7.303843
75%,2194.000000,4.000000,2.000000,93.000000,4.940000,347.00000,7.693482
max,7500.000000,16.000000,50.000000,1893.000000,5.000000,365.00000,8.922658


In [13]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/listings_clean.csv", index=False)